In [ ]:
import unicodedata
import re
import html

import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

DATA_DIR_PATH = "../data/"

In [17]:
INPUT_PATH = DATA_DIR_PATH + "raw/scrape_kompas_clean.csv"

df = pd.read_csv(INPUT_PATH)

df.head()

,link,date,title,content,keywords,category
0,https://entertainment.kompas.com/read/2026/07/...,2026-07-30 09:26:57,Deretan Aktor yang Pernah Perankan Spider-Man,"JAKARTA, KOMPAS.com- KehadiranSpider-Man: Bran...","peter parker, mcu, spider-man, tom holland",Hype
1,https://www.kompas.com/hype/read/2026/08/04/16...,2026-08-04 16:51:09,Aktor Ryotaro Shimizu Ditemukan Meninggal Duni...,"KOMPAS.com- Aktor Ryotaro Shimizu, yang karier...","ryotaro shimizu, ryotaro shimizu meninggal",Hype
2,https://entertainment.kompas.com/read/2026/07/...,2026-07-14 12:05:38,"Aktor Marvelous Mrs. Maisel, Josh Grisetti Men...","KOMPAS.com- Josh Grisetti, aktor yang dikenal ...","josh grisetti, josh grisetti meninggal, film j...",Hype
3,https://www.kompas.com/hype/read/2026/07/28/19...,2026-07-28 19:42:53,"Aktor Bittersweet Destiny, Wang Kai Ditemukan ...","KOMPAS.com- Aktor Taiwan Wang Kai, yang terken...","wang kaimu, wang kaimu drama, wang kai meninggal",Hype
4,https://www.kompas.com/hype/read/2026/08/04/11...,2026-08-04 11:52:54,"Aktor Thailand, Jorjay Phumhirun Mendadak Meni...",KOMPAS.com- Aktor muda Thailand Jorjay Phumhir...,"jorjay phumhirun meninggal, jorjay phumhirun",Hype


In [18]:
# Normalize Text
def normalize_text(text, keep_numbers = False):
    if pd.isna(text):
        return ""

    text = str(text)

    # Normalize Unicode representation first
    text = unicodedata.normalize("NFKC", text)

    # Decode HTML entities
    text = html.unescape(text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove URLs
    text = re.sub(
        r"(?:https?://|www\.)\S+",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Remove standalone domains
    text = re.sub(
        r"\b(?:[\w-]+\.)+(?:com|co\.id|id|net|org|go\.id|ac\.id)\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Remove email addresses
    text = re.sub(
        r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b",
        " ",
        text
    )

    # Remove @mentions
    text = re.sub(
        r"@[A-Za-z0-9_]+",
        " ",
        text
    )

    # Lowercase
    text = text.lower()

    # Keep Unicode letters, digits, and whitespace
    # Remove punctuation and special characters
    text = re.sub(r"[^\w\s]", " ", text)

    # Remove underscores
    text = re.sub(r"_", " ", text)

    # Optionally remove numbers
    if not keep_numbers:
        text = re.sub(r"\d+", " ", text)

    # Normalize elongated characters
    # Example: goooood -> good
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [19]:
normalized_columns = ["title", "content", "keywords"]

for column in normalized_columns:
    df[f"normalized_{column}"] = df[column].apply(normalize_text)
    print(f"{column} normalized.")

title normalized.
content normalized.
keywords normalized.


In [20]:
# Combine normalized columns into a single column
text_columns = [
    "normalized_title",
    "normalized_content",
    "normalized_keywords"
]

df["normalized"] = (
    df[text_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.strip()
)

In [21]:
print("Normalization sample:")
display(
    df.sample(
        min(10, len(df)),
        random_state=42
    )
)

Normalization sample:


,link,date,title,content,keywords,category,normalized_title,normalized_content,normalized_keywords,normalized
6170,https://money.kompas.com/read/2026/07/15/14002...,2026-07-15 14:00:21,"Pangsa Bank Syariah Masih 8 Persen, BSI Perlua...","JAKARTA, KOMPAS.com -PT Bank Syariah Indonesia...","perbankan syariah, bank syariah, bsi, bank sya...",Money,pangsa bank syariah masih persen bsi perluas j...,jakarta pt bank syariah indonesia persero tbk ...,perbankan syariah bank syariah bsi bank syaria...,pangsa bank syariah masih persen bsi perluas j...
5832,https://nasional.kompas.com/read/2026/07/22/15...,2026-07-22 15:20:01,Prabowo Lantik Sudaryono Jadi Kepala BGN Ganti...,"JAKARTA, KOMPAS.com -Presiden RI Prabowo Subia...","nanik s deyang, sudaryono, prabowo, kepala bgn...",News,prabowo lantik sudaryono jadi kepala bgn ganti...,jakarta presiden ri prabowo subianto resmi mel...,nanik s deyang sudaryono prabowo kepala bgn su...,prabowo lantik sudaryono jadi kepala bgn ganti...
286,https://bola.kompas.com/read/2026/07/26/133509...,2026-07-26 13:35:09,"Bobotoh dan Aremania Satu Tribun, Sinyal Posit...","BANDUNG, KOMPAS.com -Piala Presiden 2026 menja...","persib, persib bandung, aremania, arema, igor ...",Bola,bobotoh dan aremania satu tribun sinyal positi...,bandung piala presiden menjadi momentum kembal...,persib persib bandung aremania arema igor toli...,bobotoh dan aremania satu tribun sinyal positi...
8484,https://biz.kompas.com/read/2026/07/24/2233224...,2026-07-24 22:33:22,JKN Jadi Fondasi Perlindungan Kesehatan Anak M...,KOMPAS.com –Masa anak-anak menjadi periode pen...,"bpjs kesehatan, ispa, indonesia emas, fktp, pr...",Advertorial,jkn jadi fondasi perlindungan kesehatan anak m...,masa anak anak menjadi periode penting dalam m...,bpjs kesehatan ispa indonesia emas fktp progra...,jkn jadi fondasi perlindungan kesehatan anak m...
5684,https://nasional.kompas.com/read/2026/07/27/06...,2026-07-27 06:12:33,Partai Buruh Pilih Henry Saragih Jadi Sekjen Baru,"JAKARTA, KOMPAS.com- Ketua Umum Serikat Petani...","serikat petani indonesia, partai buruh, henry ...",News,partai buruh pilih henry saragih jadi sekjen baru,jakarta ketua umum serikat petani indonesia sp...,serikat petani indonesia partai buruh henry sa...,partai buruh pilih henry saragih jadi sekjen b...
761,https://regional.kompas.com/read/2026/07/22/14...,2026-07-22 14:34:04,BGN Tanggung Biaya Korban Keracunan Susu MBG d...,"SEMARANG, KOMPAS.com– Badan Gizi Nasional (BGN...","mbg blora, keracunan mbg blora, berita blora, ...",Regional,bgn tanggung biaya korban keracunan susu mbg d...,semarang badan gizi nasional bgn memastikan ak...,mbg blora keracunan mbg blora berita blora kpp...,bgn tanggung biaya korban keracunan susu mbg d...
3684,https://money.kompas.com/read/2026/07/12/17170...,2026-07-12 17:17:00,Perluasan Peran Koperasi: Dari Sumur Minyak hi...,"JAKARTA, KOMPAS.com -Menteri Koperasi (Menkop)...","koperasi, pengelolaan sumur minyak, koperasi s...",Money,perluasan peran koperasi dari sumur minyak hin...,jakarta menteri koperasi menkop ferry julianto...,koperasi pengelolaan sumur minyak koperasi sek...,perluasan peran koperasi dari sumur minyak hin...
7879,https://nasional.kompas.com/read/2026/07/21/09...,2026-07-21 09:33:52,Hak dan Kewajiban Ojol Akan Diatur dalam Undan...,"JAKARTA, KOMPAS.com- Wakil Ketua Badan Legisla...","ojek online, ojol, baleg, ruu llaj, ruu transp...",News,hak dan kewajiban ojol akan diatur dalam undan...,jakarta wakil ketua badan legislasi baleg dpr ...,ojek online ojol baleg ruu llaj ruu transporta...,hak dan kewajiban ojol akan diatur dalam undan...
1421,https://medan.kompas.com/read/2026/07/16/19561...,2026-07-16 19:56:10,"Potret Warga Medan Antre BBM di Tengah Banjir,...","MEDAN, KOMPAS.com— Antrean BBM yang mengular d...","bbm, spbu, pertamina, medan, antrean bbm, beri...",Regional,potret warga medan antre bbm di tengah banjir ...,medan antrean bbm yang mengular di kota medan ...,bbm spbu pertamina medan antrean bbm berita me...,potret warga me

In [23]:
# Tokenization

df["tokens"] = df["normalized"].apply(word_tokenize)

print("Dataset tokenization completed.")

Dataset tokenization completed.


In [24]:
print("\nTokenization sample:")
display(
    df[["link", "normalized", "tokens"]].head()
)


Tokenization sample:


,link,normalized,tokens
0,https://entertainment.kompas.com/read/2026/07/...,deretan aktor yang pernah perankan spider man ...,"[deretan, aktor, yang, pernah, perankan, spide..."
1,https://www.kompas.com/hype/read/2026/08/04/16...,aktor ryotaro shimizu ditemukan meninggal duni...,"[aktor, ryotaro, shimizu, ditemukan, meninggal..."
2,https://entertainment.kompas.com/read/2026/07/...,aktor marvelous mrs maisel josh grisetti menin...,"[aktor, marvelous, mrs, maisel, josh, grisetti..."
3,https://www.kompas.com/hype/read/2026/07/28/19...,aktor bittersweet destiny wang kai ditemukan m...,"[aktor, bittersweet, destiny, wang, kai, ditem..."
4,https://www.kompas.com/hype/read/2026/08/04/11...,aktor thailand jorjay phumhirun mendadak menin...,"[aktor, thailand, jorjay, phumhirun, mendadak,..."


In [25]:
# Stopword Removal
stopwords_id = set(stopwords.words("indonesian"))
stopwords_en = set(stopwords.words("english"))

stop_words = stopwords_id.union(stopwords_en)

In [28]:
def remove_stopwords(token_lists):
    result = []

    for tokens in token_lists:
        filtered_tokens = [
            token
            for token in tokens
            if token.lower() not in stop_words
        ]

        result.append(filtered_tokens)

    return result

In [29]:
df["filtered_tokens"] = remove_stopwords(df["tokens"].tolist())
print("Tokens Filtered using Stopword Removal NLTK en and id (union)")

Tokens Filtered using Stopword Removal NLTK en and id (union)


In [30]:
print("\nFiltered sample:")
display(
    df[["link", "normalized", "tokens", "filtered_tokens"]].head()
)


Filtered sample:


,link,normalized,tokens,filtered_tokens
0,https://entertainment.kompas.com/read/2026/07/...,deretan aktor yang pernah perankan spider man ...,"[deretan, aktor, yang, pernah, perankan, spide...","[deretan, aktor, perankan, spider, man, jakart..."
1,https://www.kompas.com/hype/read/2026/08/04/16...,aktor ryotaro shimizu ditemukan meninggal duni...,"[aktor, ryotaro, shimizu, ditemukan, meninggal...","[aktor, ryotaro, shimizu, ditemukan, meninggal..."
2,https://entertainment.kompas.com/read/2026/07/...,aktor marvelous mrs maisel josh grisetti menin...,"[aktor, marvelous, mrs, maisel, josh, grisetti...","[aktor, marvelous, mrs, maisel, josh, grisetti..."
3,https://www.kompas.com/hype/read/2026/07/28/19...,aktor bittersweet destiny wang kai ditemukan m...,"[aktor, bittersweet, destiny, wang, kai, ditem...","[aktor, bittersweet, destiny, wang, kai, ditem..."
4,https://www.kompas.com/hype/read/2026/08/04/11...,aktor thailand jorjay phumhirun mendadak menin...,"[aktor, thailand, jorjay, phumhirun, mendadak,...","[aktor, thailand, jorjay, phumhirun, mendadak,..."


In [31]:
# Save the normalized dataset
OUTPUT_PATH = DATA_DIR_PATH + "preprocessed/normalized_dataset.csv"

df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"Normalized dataset saved to {OUTPUT_PATH}")

Normalized dataset saved to ../data/preprocessed/normalized_dataset.csv
